In [0]:
%sql
use catalog cat_ltm10877547;
use schema schema_data_integration;
show schemas in cat_ltm10877547;

databaseName
bronze
default
gold
information_schema
schema_data_integration
silver
training_assessment_delivery_ops
training_assessment_hospitality


task 2:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
schema=StructType([
    StructField("order_id",StringType(),True),
    StructField("city",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(12,2),True),
    StructField("order_status",StringType(),True),
    StructField("order_date",DateType(),True)
])
df=spark.read.csv("/Volumes/cat_ltm10877547/schema_data_integration/ext_loc_soumiya/assessments/10877547/e01/E01_Marketplace_Order_Quality_and_Revenue.csv",header=True,schema=schema)

In [0]:
display(df.count())
display(df.filter(
    (col("order_status")=="CANCELLED") |
    (col("quantity")<=0) |
    (col("unit_price").isNull())
))
display(df)

20

order_id,city,category,quantity,unit_price,order_status,order_date
O004,Delhi,Fashion,1,150,CANCELLED,2026-09-05
O015,Chennai,Electronics,4,1250,CANCELLED,2026-09-01
O018,Mumbai,Home,3,null,COMPLETED,2026-09-04


order_id,city,category,quantity,unit_price,order_status,order_date
O001,Pune,Books,2,350,COMPLETED,2026-09-02
O002,Mumbai,Home,3,700,COMPLETED,2026-09-03
O003,Chennai,Electronics,4,1250,COMPLETED,2026-09-04
O004,Delhi,Fashion,1,150,CANCELLED,2026-09-05
O005,Pune,Books,2,350,COMPLETED,2026-09-06
O006,Mumbai,Home,3,700,COMPLETED,2026-09-07
O007,Chennai,Electronics,4,1250,COMPLETED,2026-09-08
O008,Delhi,Fashion,1,150,COMPLETED,2026-09-09
O009,Pune,Books,0,350,COMPLETED,2026-09-10
O010,Mumbai,Home,3,700,COMPLETED,2026-09-11


In [0]:
clean_df=df.filter(
    (col("order_status")=="COMPLETED") &
    (col("quantity")>0) &
    (col("unit_price").isNotNull())
)
display(clean_df.count())
display(clean_df)

16

order_id,city,category,quantity,unit_price,order_status,order_date
O001,Pune,Books,2,350,COMPLETED,2026-09-02
O002,Mumbai,Home,3,700,COMPLETED,2026-09-03
O003,Chennai,Electronics,4,1250,COMPLETED,2026-09-04
O005,Pune,Books,2,350,COMPLETED,2026-09-06
O006,Mumbai,Home,3,700,COMPLETED,2026-09-07
O007,Chennai,Electronics,4,1250,COMPLETED,2026-09-08
O008,Delhi,Fashion,1,150,COMPLETED,2026-09-09
O010,Mumbai,Home,3,700,COMPLETED,2026-09-11
O011,Chennai,Electronics,4,1250,COMPLETED,2026-09-12
O012,Delhi,Fashion,1,150,COMPLETED,2026-09-13


task 3:

In [0]:
clean_df=clean_df.withColumn("order_value",col("quantity").cast("decimal(14,2)")*col("unit_price").cast("decimal(14,2)"))
clean_df=clean_df.withColumn("value_band",when(col("order_value")>=1500.00,"HIGH").otherwise("STANDARD"))
display(clean_df.columns)
clean_df.write.format("delta").mode("overwrite").saveAsTable("cat_ltm10877547.schema_data_integration.marketplace_orders_silver")

_1
order_id
city
category
quantity
unit_price
order_status
order_date
order_value
value_band


In [0]:
display(spark.sql("describe detail cat_ltm10877547.schema_data_integration.marketplace_orders_silver"))
display(spark.sql("select count(*) as order_count,sum(order_value) as total_value,sum(case when value_band='HIGH' then 1 else 0 end) as high_value_orders from cat_ltm10877547.schema_data_integration.marketplace_orders_silver group by city order by city"))
display(spark.sql("select count(*) as total_count,sum(order_value) as total_revenue from cat_ltm10877547.schema_data_integration.marketplace_orders_silver"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7e83831e-40b8-43a6-a338-e3de5eab0b2a,cat_ltm10877547.schema_data_integration.marketplace_orders_silver,null,abfss://unity-catalog-storage@dbstoragekfow3wgm4zqhk.dfs.core.windows.net/7405612971153131/soumiya/__unitystorage/schemas/bf8b1487-1e5d-4bda-9e81-6372c2790bab/tables/51a91e4c-a66f-4d39-8abc-2fb2ed40492e,2026-10-17T13:16:03.110Z,2026-10-17T13:16:04.000Z,List(),List(),1,3083,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 51a91e4c-a66f-4d39-8abc-2fb2ed40492e)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


order_count,total_value,high_value_orders
4,20000.00,4
4,600.00,0
4,8400.00,4
4,2800.00,0


total_count,total_revenue
16,31800.00


In [0]:
display(clean_df.agg(count("*").alias("total_count"),sum("order_value").alias("total_revenue")))

total_count,total_revenue
16,31800.00
